In [1]:
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score, make_scorer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report

In [2]:
# 7 = 5 PCs but with baseline
# 9 = 5 PCs but with 4 clusters
# else = num PC

d5_coords = np.load(f"/Users/hades/Desktop/Bruchas Lab/Encoder_Decoder/DJM_binary_classification/Cluster_Validation/cluster_data/d1_clusters/D5_PC_coordinates.npy")
d1_coords = np.load(f"/Users/hades/Desktop/Bruchas Lab/Encoder_Decoder/DJM_binary_classification/Cluster_Validation/cluster_data/d1_clusters/D5_PC_coordinates.npy")
cluster_labels = np.load(f"/Users/hades/Desktop/Bruchas Lab/Encoder_Decoder/DJM_binary_classification/Cluster_Validation/cluster_data/d1_clusters/cluster_labels.npy")

X_train, X_test, y_train, y_test = train_test_split(
    d1_coords,
    cluster_labels,
    train_size=0.80,
    test_size=0.20,
    random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [3]:
# Define the parameter grid
param_grid = {
    'C': [0.01, 0.1, 1, 10, 100],
    'kernel': ['rbf', 'linear', 'poly'],
    'gamma': ['scale', 'auto', 0.1, 1],
    'class_weight': [None, 'balanced'],
    'degree': [2, 3, 4, 5],  # for poly kernel
    'coef0': [0, 0.1, 0.5, 1],  # for poly kernel
}

# Create an SVM classifier
svm = SVC(random_state=42)

# Define a custom scorer that uses F1 score
f1_scorer = make_scorer(f1_score, average='weighted')

# Set up GridSearchCV with StratifiedKFold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
random_search = RandomizedSearchCV(svm, param_grid, n_iter=100, cv=skf, 
                                   scoring=f1_scorer, n_jobs=-1, random_state=42)

# Perform the grid search
random_search.fit(X_train_scaled, y_train)

"""
10 - {'kernel': 'rbf', 'gamma': 'scale', 'degree': 5, 'coef0': 0.1, 'class_weight': None, 'C': 10} - 0.7988251282948253
5  - {'kernel': 'linear', 'class_weight': None, 'C': 10} - 0.8545217397255016
3  - {'kernel': 'linear', 'class_weight': None, 'C': 10} - 0.821943847398393
"""

# Print the best parameters and score
print("Best parameters:", random_search.best_params_)
print("Best cross-validation score:", random_search.best_score_)

Best parameters: {'kernel': 'poly', 'gamma': 'auto', 'degree': 3, 'coef0': 1, 'class_weight': None, 'C': 10}
Best cross-validation score: 0.3357053603106235


In [4]:
# Use the best model to make predictions
# best_model = random_search.best_estimator_

best_model = SVC(C=100, kernel='linear', class_weight=None).fit(X_train_scaled, y_train)    # Based on previous best
d1_pred = best_model.predict(X_test_scaled)

d1_accuracy = accuracy_score(y_test, d1_pred)
d1_f1 = f1_score(y_test, d1_pred, average='weighted')

print("Day 1 Results")
print(classification_report(y_test, d1_pred))
print('Accuracy: ', "%.2f" % (d1_accuracy*100))
print('F1: ', "%.2f" % (d1_f1*100))

Day 1 Results
              precision    recall  f1-score   support

           1       0.00      0.00      0.00         6
           2       0.00      0.00      0.00         6
           3       0.55      1.00      0.71        23
           4       0.00      0.00      0.00         7

    accuracy                           0.55        42
   macro avg       0.14      0.25      0.18        42
weighted avg       0.30      0.55      0.39        42

Accuracy:  54.76
F1:  38.75


/Users/hades/Desktop/Bruchas Lab/Encoder_Decoder/DJM_binary_classification/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/hades/Desktop/Bruchas Lab/Encoder_Decoder/DJM_binary_classification/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/hades/Desktop/Bruchas Lab/Encoder_Decoder/DJM_binary_classification/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in label

In [5]:
d5_coords_scaled = scaler.transform(d5_coords)
d5_pred = best_model.predict(d5_coords_scaled)

d5_accuracy = accuracy_score(cluster_labels, d5_pred)
d5_f1 = f1_score(cluster_labels, d5_pred, average='weighted')

print("Day 5 Results")
print(classification_report(cluster_labels, d5_pred))
print('Accuracy: ', "%.2f" % (d5_accuracy*100))
print('F1: ', "%.2f" % (d5_f1*100))

Day 5 Results
              precision    recall  f1-score   support

           1       0.00      0.00      0.00        36
           2       0.00      0.00      0.00        33
           3       0.47      1.00      0.64        96
           4       0.00      0.00      0.00        41

    accuracy                           0.47       206
   macro avg       0.12      0.25      0.16       206
weighted avg       0.22      0.47      0.30       206

Accuracy:  46.60
F1:  29.63


/Users/hades/Desktop/Bruchas Lab/Encoder_Decoder/DJM_binary_classification/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/hades/Desktop/Bruchas Lab/Encoder_Decoder/DJM_binary_classification/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/hades/Desktop/Bruchas Lab/Encoder_Decoder/DJM_binary_classification/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in label

In [6]:
np.random.seed(42)

shuffled_labels = np.random.uniform(1, 5, 206).astype(int)
X_train_shuffled, X_test_shuffled, y_train_shuffled, y_test_shuffled = train_test_split(
    d1_coords,
    shuffled_labels,
    train_size=0.80,
    test_size=0.20,
    random_state=42)

X_train_shuffled_scaled = scaler.fit_transform(X_train_shuffled)
X_test_shuffled_scaled = scaler.transform(X_test_shuffled)

In [7]:
shuffled_model = SVC(C=10, kernel='linear', class_weight=None).fit(X_train_shuffled_scaled, y_train_shuffled)
d1_shuffled_pred = shuffled_model.predict(X_test_shuffled_scaled)

d1_shuffled_accuracy = accuracy_score(y_test_shuffled, d1_shuffled_pred)
d1_shuffled_f1 = f1_score(y_test_shuffled, d1_shuffled_pred, average='weighted')

print("Shuffled Day 1 Results")
print(classification_report(y_test_shuffled, d1_shuffled_pred))
print('Accuracy: ', "%.2f" % (d1_shuffled_accuracy*100))
print('F1: ', "%.2f" % (d1_shuffled_f1*100))


Shuffled Day 1 Results
              precision    recall  f1-score   support

           1       0.19      0.50      0.28         8
           2       0.00      0.00      0.00        11
           3       1.00      0.27      0.43        11
           4       0.28      0.42      0.33        12

    accuracy                           0.29        42
   macro avg       0.37      0.30      0.26        42
weighted avg       0.38      0.29      0.26        42

Accuracy:  28.57
F1:  26.00


/Users/hades/Desktop/Bruchas Lab/Encoder_Decoder/DJM_binary_classification/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/hades/Desktop/Bruchas Lab/Encoder_Decoder/DJM_binary_classification/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/hades/Desktop/Bruchas Lab/Encoder_Decoder/DJM_binary_classification/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in label

In [8]:
d5_coords_scaled = scaler.transform(d5_coords)
d5_pred = shuffled_model.predict(d5_coords_scaled)

d5_accuracy = accuracy_score(cluster_labels, d5_pred)
d5_f1 = f1_score(cluster_labels, d5_pred, average='weighted')

print("Shuffled Day 5 Results")
print(classification_report(cluster_labels, d5_pred))
print('Accuracy: ', "%.2f" % (d5_accuracy*100))
print('F1: ', "%.2f" % (d5_f1*100))

Shuffled Day 5 Results
              precision    recall  f1-score   support

           1       0.15      0.53      0.24        36
           2       0.50      0.03      0.06        33
           3       0.38      0.08      0.14        96
           4       0.15      0.22      0.18        41

    accuracy                           0.18       206
   macro avg       0.30      0.22      0.15       206
weighted avg       0.31      0.18      0.15       206

Accuracy:  17.96
F1:  15.02


In [9]:
np.random.seed(42)

shuffled_labels = np.random.permutation(cluster_labels)
X_train_shuffled, X_test_shuffled, y_train_shuffled, y_test_shuffled = train_test_split(
    d1_coords,
    shuffled_labels,
    train_size=0.80,
    test_size=0.20,
    random_state=42)

X_train_shuffled_scaled = scaler.fit_transform(X_train_shuffled)
X_test_shuffled_scaled = scaler.transform(X_test_shuffled)

In [10]:
shuffled_model = SVC(C=10, kernel='linear', class_weight=None).fit(X_train_shuffled_scaled, y_train_shuffled)
d1_shuffled_pred = shuffled_model.predict(X_test_shuffled_scaled)

d1_shuffled_accuracy = accuracy_score(y_test_shuffled, d1_shuffled_pred)
d1_shuffled_f1 = f1_score(y_test_shuffled, d1_shuffled_pred, average='weighted')

print("Uniform Labeling Day 1 Results")
print(classification_report(y_test_shuffled, d1_shuffled_pred))
print('Accuracy: ', "%.2f" % (d1_shuffled_accuracy*100))
print('F1: ', "%.2f" % (d1_shuffled_f1*100))

Uniform Labeling Day 1 Results
              precision    recall  f1-score   support

           1       0.00      0.00      0.00        11
           2       0.00      0.00      0.00         3
           3       0.48      1.00      0.65        20
           4       0.00      0.00      0.00         8

    accuracy                           0.48        42
   macro avg       0.12      0.25      0.16        42
weighted avg       0.23      0.48      0.31        42

Accuracy:  47.62
F1:  30.72


/Users/hades/Desktop/Bruchas Lab/Encoder_Decoder/DJM_binary_classification/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/hades/Desktop/Bruchas Lab/Encoder_Decoder/DJM_binary_classification/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/hades/Desktop/Bruchas Lab/Encoder_Decoder/DJM_binary_classification/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in label

In [11]:
d5_coords_scaled = scaler.transform(d5_coords)
d5_pred = shuffled_model.predict(d5_coords_scaled)

d5_accuracy = accuracy_score(cluster_labels, d5_pred)
d5_f1 = f1_score(cluster_labels, d5_pred, average='weighted')

print("Uniform Labeling Day 1 Results")
print(classification_report(cluster_labels, d5_pred))
print('Accuracy: ', "%.2f" % (d5_accuracy*100))
print('F1: ', "%.2f" % (d5_f1*100))

Uniform Labeling Day 1 Results
              precision    recall  f1-score   support

           1       0.00      0.00      0.00        36
           2       0.00      0.00      0.00        33
           3       0.47      1.00      0.64        96
           4       0.00      0.00      0.00        41

    accuracy                           0.47       206
   macro avg       0.12      0.25      0.16       206
weighted avg       0.22      0.47      0.30       206

Accuracy:  46.60
F1:  29.63


/Users/hades/Desktop/Bruchas Lab/Encoder_Decoder/DJM_binary_classification/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/hades/Desktop/Bruchas Lab/Encoder_Decoder/DJM_binary_classification/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/hades/Desktop/Bruchas Lab/Encoder_Decoder/DJM_binary_classification/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in label